#### Introduction
In the previous example, there was a flaw in the code. In that code, we were doing batch gradient descent, that calculates the gradient of the cost function using the entire training dataset before updating the model's parameters exactly once per iteration. There are main two drawbacks in this process.
 - **Memory Inefficient:** For each iteration, the dataset is loaded in ram and then processed for gradient descent. It works fine for a small training datasets, but imagine what will happen if there are millions of tons of dataset?
 -  **Better Convergence:** In batch gradient descent, convergence is inefficient. We can not reach in good solution or parameters quickly in batch gradient descent. It is getting the full dataset at once and optimizes, that makes it to converge slow. Moreover, it lacks randomness, each time, for same input data it is making the optimization. So it stucks somewhere between.

**The solution:** Make the whole dataset into small chuncks or batches, then feed them to the model in different iterations. It is called **Mini-Batch GD**

#### Dataset and Dataloader classes:
Dataset and DataLoader are core abstractions in Pytorch that decouple how you define your data from how efficiently iterate over it in training loops. 

##### High-level overview:
- 3 components: Data, Dataset Class, DataLoader class
- Loop: Loads data from the source, divide it into batches, insert it in the training loop
  - Data lies in memory. We need to load it from the memory.
  - Dataset class loads it from the memory
    - Dataset class knows the momory space of the data
    - It can load data rows one by one effectively
  - DataLoader Class: Main component
    - It does the batching part.
    - DataLoader decides in each batch how many rows will be there.
  - DataLoader class asks Dataset class to process the rows from memory
  - Upon the trigger from DataLoader, Dataset class fetches rows from the memory
  - DataLoader loads tha data provided by Dataset class and make batches using them
  - Then it inserts the loaded data into batches in the training loop

#### Class examples
##### Dataset Class
The dataset class is essentially a blueprint. When you create a custom Dataset, you decide how data is loaded and returned. We need to define and implement those following methods:
- **`__init__()`:** which tells how data should be loaded. We will write underlying logics for loading data in this constructor method.
- **`__len__()`:** which returns the total number of samples (rows)
- **`__getItem__(index)`:** which returns the data (and label) at the given index (row)

##### DataLoader Class
The DataLoader class wraps a Datset and handles batching, shuffling, and parallel loading.

##### DataLoader Control Flow:
- At start of each epoch, the DataLoader (if shuffle=True) shuffles indices.
- It divides the indices into chunks of batch_size.
- For each index in the chunk, data samples are fetched from the Dataset object
- The samples are then collected and combined into a batch (using collate_fn)
- The batch is returned to the main training loop

In [2]:
from sklearn.datasets import make_classification
import torch

In [5]:
# Step 1: Create a synthetic classification dataset using sklearn - random feature vectors (X) with class labels (y)
X, y = make_classification(
    n_samples=10, # number of samples/rows or data points, default 100
    n_features=2, # number of total columns/features per sample
    n_informative=2, # number of features that actually carry signal about the class (the rest are noise/combinations) must be <= n_features
    n_redundant=0, # number of redundant features
    n_classes=2, # number of target classes/labels, default 2 (binary classification)
    random_state=42 # Seed for reproducibility - same seed gives the same dataset each time
)
# n_repeated = number of duplicated features (copies of informative/redundant ones)
# n_clusters_per_class = How many clusters each class is split into in feature space - more clusters = harder decision boundary
# n_informative + n_redundant + n_repeated must be ≤ n_features, 
# otherwise it throws an error, 
# since it can't fit more "meaningful" columns than total columns available.

In [8]:
print(X)
print(X.shape)

[[ 1.06833894 -0.97007347]
 [-1.14021544 -0.83879234]
 [-2.8953973   1.97686236]
 [-0.72063436 -0.96059253]
 [-1.96287438 -0.99225135]
 [-0.9382051  -0.54304815]
 [ 1.72725924 -1.18582677]
 [ 1.77736657  1.51157598]
 [ 1.89969252  0.83444483]
 [-0.58723065 -1.97171753]]
(10, 2)


In [9]:
print(y)
print(y.shape)

[1 0 0 0 0 1 1 1 1 0]
(10,)


In [10]:
# convert the data into tensors
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

In [11]:
from torch.utils.data import Dataset, DataLoader